In [2]:
import pandas as pd 
import requests, json


In [3]:
# Load full dataset with predictions
energy_df = pd.read_csv("data/energy_source_with_predictions.csv")
# Load test results with errors
rf_results = pd.read_csv("data/rf_results.csv")

In [4]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': energy_df.nunique(),
    'Data Type': energy_df.dtypes,
    'Missing Values (Total)': energy_df.isnull().sum(),
    'Missing Values (%)': (energy_df.isnull().sum() / len(energy_df)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

                        Unique Values Data Type  Missing Values (Total)  \
Date                             2922    object                       0   
region_name                        12    object                       0   
RR                              26133   float64                       0   
ALTI                            18843   float64                       0   
PMERM                           17111   float64                       0   
INST                            23980   float64                       0   
GLOT                            29533   float64                       0   
DHUMI40                         16209   float64                       0   
NEIGETOTX                        9070   float64                       0   
NEIG                             1211   float64                       0   
BROU                             1158   float64                       0   
ORAG                             1029   float64                       0   
GRESIL                   

In [5]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': rf_results.nunique(),
    'Data Type': rf_results.dtypes,
    'Missing Values (Total)': rf_results.isnull().sum(),
    'Missing Values (%)': (rf_results.isnull().sum() / len(rf_results)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

              Unique Values Data Type  Missing Values (Total)  \
Date                   2655    object                       0   
region_name              12    object                       0   
RR                     5887   float64                       0   
ALTI                   4574   float64                       0   
PMERM                  5294   float64                       0   
INST                   5843   float64                       0   
GLOT                   6245   float64                       0   
DHUMI40                3515   float64                       0   
NEIGETOTX              2129   float64                       0   
BROU                    639   float64                       0   
ORAG                    454   float64                       0   
GRESIL                   22   float64                       0   
GRELE                   136   float64                       0   
ROSEE                   119   float64                       0   
VERGLAS                  

In [6]:
print(energy_df["region_name"].unique())

['Auvergne-Rhône-Alpes' 'Bourgogne-Franche-Comté' 'Bretagne'
 'Centre-Val de Loire' 'Grand Est' 'Hauts-de-France' 'Normandie'
 'Nouvelle-Aquitaine' 'Occitanie' 'Pays de la Loire'
 "Provence-Alpes-Côte d'Azur" 'Île-de-France']


In [7]:
url = 'https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson'
r = requests.get(url)
geojson = r.json()

print(f"Total features: {len(geojson['features'])}")
print(geojson['features'][0]['properties'])

Total features: 13
{'code': '11', 'nom': 'Île-de-France'}


### Methodological note — error metrics

The **Absolute Percentage Error (APE)** is computed at the observation level as:

$$APE = \\frac{|y_{true} - y_{pred}|}{y_{true} + \\epsilon}$$

where $\\epsilon = 10^{-8}$ is a small constant added to avoid division by zero. 
Its effect is negligible given that consumption values are in the tens of kWh per capita.

**Example:** for $y_{true} = 40.0$ and $y_{pred} = 42.0$:

| Metric | Value |
|:-------|------:|
| Error ($y_{true} - y_{pred}$) | −2.0 |
| Absolute error | 2.0 |
| APE | 0.05 → 5% |

The **regional MAPE** is the mean of all individual APEs for test observations 
belonging to that region:

$$MAPE_{region} = \\frac{1}{n} \\sum_{i=1}^{n} APE_i$$

All error metrics are computed **on the held-out test set only** (`rf_results` derives 
from `X_test` / `y_test`). This ensures we measure generalization to unseen data, 
not in-sample fit.

In [9]:
tooltip_data = rf_results.groupby('region_name').agg(
    mean_mape      = ('ape_rf',       'mean'),
    mean_mae       = ('abs_error_rf', 'mean'),
    mean_predicted = ('y_pred_rf',    'mean'),
    mean_true      = ('y_true',       'mean'),
    mean_bias      = ('error_rf',     'mean'),
    density        = ('density',      'mean'),
    n_obs          = ('y_true',       'count')
).round(3).reset_index()

# Convert to dict keyed by region name — easy to look up in D3
metrics_dict = tooltip_data.set_index('region_name').to_dict(orient='index')

with open('data/region_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics_dict, ensure_ascii=False, indent=2))

{
  "Auvergne-Rhône-Alpes": {
    "mean_mape": 0.04,
    "mean_mae": 1.8,
    "mean_predicted": 45.427,
    "mean_true": 45.352,
    "mean_bias": -0.076,
    "density": 112.586,
    "n_obs": 584
  },
  "Bourgogne-Franche-Comté": {
    "mean_mape": 0.051,
    "mean_mae": 2.002,
    "mean_predicted": 41.075,
    "mean_true": 40.954,
    "mean_bias": -0.121,
    "density": 58.586,
    "n_obs": 523
  },
  "Bretagne": {
    "mean_mape": 0.045,
    "mean_mae": 1.609,
    "mean_predicted": 36.954,
    "mean_true": 36.506,
    "mean_bias": -0.448,
    "density": 121.641,
    "n_obs": 569
  },
  "Centre-Val de Loire": {
    "mean_mape": 0.049,
    "mean_mae": 1.914,
    "mean_predicted": 39.756,
    "mean_true": 39.849,
    "mean_bias": 0.093,
    "density": 65.3,
    "n_obs": 324
  },
  "Grand Est": {
    "mean_mape": 0.042,
    "mean_mae": 1.797,
    "mean_predicted": 44.569,
    "mean_true": 44.519,
    "mean_bias": -0.05,
    "density": 96.29,
    "n_obs": 581
  },
  "Hauts-de-France": {
  